# Model Comparison for Relapse-Free Survival (RFS) Prediction

## Overview

This notebook compares the proposed **Multimodal Transformer** with traditional survival analysis models for **Relapse-Free Survival (RFS) prediction** using the TCGA-BRCA multimodal dataset.

The objective is to evaluate whether multimodal Transformer-based learning provides improved predictive performance compared with conventional machine learning and statistical survival models.

The comparison includes three categories of models:

- **Multimodal Transformer** (proposed model)
- **Cox Proportional Hazards (CoxPH)**
- **Random Survival Forest (RSF)**

In addition, pathology-only neural network baselines are evaluated to investigate the predictive value of pathology image representations.

---

# Dataset

The evaluation uses the previously constructed RFS dataset containing:

- Clinical features
- Genomic features
- Histopathology image features
- Relapse-Free Survival outcomes

The same independent test set used throughout the study is employed to ensure a fair comparison across all models.

---

# Models Evaluated

## 1. Multimodal Transformer

The proposed Transformer integrates three complementary modalities:

- Clinical features
- Genomic features
- Histopathology image features

A pretrained checkpoint from the previous experiment is loaded, and survival risk predictions are generated for the test cohort.

---

## 2. Cox Proportional Hazards (CoxPH)

CoxPH models are trained and evaluated using different combinations of input modalities:

- Clinical only
- Genomic only
- Clinical + Genomic
- Clinical + Genomic + Pathology
- PCA-compressed pathology only

Continuous features are standardized prior to model fitting, and constant features are removed automatically.

---

## 3. Random Survival Forest (RSF)

Random Survival Forest models are evaluated using the same modality configurations:

- Clinical only
- Genomic only
- Clinical + Genomic
- Clinical + Genomic + Pathology
- Clinical + Genomic + PCA Pathology
- PCA-compressed pathology only

---

## 4. Pathology-Only Neural Network Baselines

To further investigate the predictive ability of pathology features alone, two multilayer perceptron (MLP) models are evaluated:

- Pathology MLP using the original pathology embeddings
- PCA Pathology MLP using PCA-reduced pathology features

These models provide additional deep learning baselines for comparison with the proposed multimodal Transformer.

---

# Evaluation Metric

All models are evaluated using the **Concordance Index (C-index)**, which measures how well predicted risk scores rank patients according to their relapse-free survival outcomes.

A higher C-index indicates better discrimination between patients with lower and higher recurrence risk.

---

# Final Comparison

The notebook produces a unified comparison table summarizing the predictive performance of all evaluated models.

The proposed **Multimodal Transformer** achieved the highest performance among all evaluated approaches.

| Model | Modalities | C-index |
|-------|------------|---------:|
| Multimodal Transformer | Clinical + Genomic + Pathology | **0.8228** |
| CoxPH | Clinical | 0.7532 |
| RSF | Clinical | 0.7437 |
| CoxPH | Clinical + Genomic | 0.7057 |
| CoxPH | Clinical + Genomic + Pathology | 0.6646 |
| RSF | Clinical + Genomic | 0.6551 |
| CoxPH | PCA Pathology Only | 0.5823 |
| RSF | PCA Pathology Only | 0.5316 |
| PCA Pathology MLP | PCA Pathology Only | 0.3323 |
| RSF | Clinical + Genomic + PCA Pathology | 0.3259 |
| Pathology MLP | Pathology Only | 0.2785 |
| RSF | Clinical + Genomic + Pathology | 0.2658 |

---

# Conclusion

This notebook demonstrates that integrating **clinical**, **genomic**, and **histopathology** information within the proposed **Multimodal Transformer** provides superior predictive performance compared with classical survival analysis methods and pathology-only neural network baselines. The results support the effectiveness of multimodal Transformer architectures for Relapse-Free Survival prediction in breast cancer patients.

##Imports

In [2]:
# ============================================================
# FIX PACKAGE VERSIONS
# ============================================================

!pip uninstall -y scikit-survival scikit-learn

!pip install -q scikit-learn==1.4.2

!pip install -q scikit-survival==0.23.1

!pip install -q lifelines


print("Packages installed")

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 34.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 6.9

In [3]:
# ============================================================
# CELL 1
# INSTALL + IMPORTS
# ============================================================




import os
import glob

import numpy as np
import pandas as pd


import torch
from torch.utils.data import Dataset, DataLoader


import torch.nn as nn


from lifelines.utils import concordance_index


from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored


import matplotlib.pyplot as plt


print("="*70)
print("IMPORTS COMPLETED")
print("="*70)

IMPORTS COMPLETED


##Mount Google Drive

In [4]:
# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive


drive.mount("/content/drive")


print("="*70)
print("DRIVE MOUNTED")
print("="*70)

Mounted at /content/drive
DRIVE MOUNTED


##Paths

In [5]:
# ============================================================
# PATH CONFIGURATION
# ============================================================


BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"


RFS_DIR = os.path.join(
    BASE_DIR,
    "rfs_feature_selection"
)


FEATURE_DIR = os.path.join(
    BASE_DIR,
    "Features"
)


MODEL_DIR = os.path.join(
    BASE_DIR,
    "trained_models"
)


PATHOLOGY_DIR = FEATURE_DIR


CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "best_transformer_all_Modailates_RFS2.pt"
)


print("="*70)
print("PATHS")
print("="*70)


print("BASE:", BASE_DIR)

print("RFS:", RFS_DIR)

print("FEATURES:", FEATURE_DIR)

print("CHECKPOINT:", CHECKPOINT_PATH)

PATHS
BASE: /content/drive/MyDrive/TCGA_BRCA
RFS: /content/drive/MyDrive/TCGA_BRCA/rfs_feature_selection
FEATURES: /content/drive/MyDrive/TCGA_BRCA/Features
CHECKPOINT: /content/drive/MyDrive/TCGA_BRCA/trained_models/best_transformer_all_Modailates_RFS2.pt


##Load Test Dataset

In [6]:
# ============================================================
# LOAD TEST DATA
# ============================================================


test_df = pd.read_csv(

    os.path.join(
        RFS_DIR,
        "rfs_test_selected.csv"
    )

)


print("="*70)
print("RFS TEST DATA")
print("="*70)


print(
    "Shape:",
    test_df.shape
)


display(
    test_df.head()
)


print(
    "Patients:",
    test_df["patient_id"].nunique()
)

RFS TEST DATA
Shape: (54, 224)


,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,histological_type_infiltratinglobularcarcinoma,histological_type_medullarycarcinoma,...,HAPLN1,RNF183,LOC96610,TMPRSS4,C6orf141,C20orf85,patient_id,file_uuid,event,survival_time
0,0.717579,0.130262,3,2,3,0,1.828816,1,True,False,...,-1.306939,-0.512511,-0.574521,0.916788,0.432721,-0.514648,TCGA-AR-A5QN,88e15db1-54ef-43e8-91d6-bb41d9fb3cf1,0,1013.0
1,0.259470,-0.329932,2,2,0,0,-0.502090,1,False,False,...,0.247674,0.088457,-0.658618,0.245905,-0.809119,-0.514648,TCGA-BH-A0DE,0bd7ca22-a281-427c-a5ef-b59b5dd02da3,0,2372.0
2,-1.572967,0.625970,3,3,1,0,0.896453,1,False,False,...,-0.174433,0.258773,1.633520,0.803308,1.029709,-0.313943,TCGA-B6-A0IE,20d69f17-d957-4dc2-ae29-3f28a9518868,1,1993.0
3,-0.962155,-0.614039,2,2,1,0,-0.035909,1,False,False,...,-0.024339,0.841921,0.888047,-0.482192,1.047505,-0.514648,TCGA-GM-A2DA,ac8c09f2-2fd6-43e2-8233-00dcf54dcb0b,1,6593.0
4,-1.496616,1.149053,2,2,1,0,0.197181,1,False,False,...,-1.167379,-0.050998,0.512568,1.291236,1.465347,-0.300371,TCGA-OL-A66O,1c8bf463-54da-4f25-a146-e9ecb3df3af1,0,528.0


Patients: 54


##Feature Groups

In [7]:
# ============================================================
# FEATURE GROUPS
# ============================================================


exclude_columns = [

    "patient_id",
    "file_uuid",
    "event",
    "survival_time"

]



clinical_features = [

    "years_to_birth",

    "Tumor_purity",

    "pathologic_stage",

    "pathology_T_stage",

    "pathology_N_stage",

    "pathology_M_stage",

    "number_of_lymph_nodes",

    "radiation_therapy"

]



clinical_features += [

    c

    for c in test_df.columns

    if c.startswith(
        (
            "histological_type",
            "PAM50",
            "race",
            "ethnicity"
        )
    )

]



genomic_features = [

    c

    for c in test_df.columns

    if c not in clinical_features
    and c not in exclude_columns

]



clinical_dim = len(
    clinical_features
)


genomic_dim = len(
    genomic_features
)



print("="*70)

print(
    "Clinical features:",
    clinical_dim
)


print(
    "Genomic features:",
    genomic_dim
)


print(
    "Pathology feature dimension: 512"
)


print("="*70)

Clinical features: 20
Genomic features: 200
Pathology feature dimension: 512


##Load Pathology Metadata

In [8]:
# ============================================================
# PATHOLOGY FEATURE LOADING
# ============================================================


MAX_PATCHES = 512



def load_pathology_features(
    file_uuid,
    max_patches=512
):


    path = os.path.join(
        PATHOLOGY_DIR,
        file_uuid + ".npy"
    )


    features = np.load(path)



    # ----------------------------
    # More patches than maximum
    # ----------------------------

    if features.shape[0] > max_patches:


        indices = np.random.choice(

            features.shape[0],

            max_patches,

            replace=False

        )


        features = features[indices]


        mask = np.ones(
            max_patches,
            dtype=np.float32
        )



    # ----------------------------
    # Padding
    # ----------------------------

    else:


        current = features.shape[0]


        padding = np.zeros(

            (
                max_patches-current,
                features.shape[1]
            ),

            dtype=np.float32

        )


        features = np.concatenate(

            [
                features,
                padding
            ],

            axis=0

        )



        mask = np.concatenate(

            [

                np.ones(
                    current,
                    dtype=np.float32
                ),


                np.zeros(
                    max_patches-current,
                    dtype=np.float32
                )

            ]

        )



    return (

        features.astype(np.float32),

        mask.astype(np.float32)

    )



print("Pathology loader created")

Pathology loader created



##Pytorch Test Dataset + Dataloader

In [9]:
# ============================================================
# DATASET + TEST LOADER
# ============================================================



class MultimodalBreastCancerDataset(Dataset):


    def __init__(
        self,
        dataframe
    ):

        self.data = dataframe.reset_index(
            drop=True
        )



    def __len__(self):

        return len(
            self.data
        )



    def __getitem__(
        self,
        idx
    ):


        row = self.data.iloc[idx]



        clinical = row[
            clinical_features
        ].values.astype(
            np.float32
        )



        genomic = row[
            genomic_features
        ].values.astype(
            np.float32
        )



        pathology, mask = load_pathology_features(

            row["file_uuid"],

            MAX_PATCHES

        )



        survival = np.float32(
            row["survival_time"]
        )


        event = np.float32(
            row["event"]
        )



        return {


            "clinical":
            torch.tensor(
                clinical
            ),



            "genomic":
            torch.tensor(
                genomic
            ),



            "pathology":
            torch.tensor(
                pathology
            ),



            "mask":
            torch.tensor(
                mask
            ),



            "survival_time":
            torch.tensor(
                survival
            ),



            "event":
            torch.tensor(
                event
            ),



            "patient_id":
            row["patient_id"]


        }





test_dataset = MultimodalBreastCancerDataset(
    test_df
)



test_loader = DataLoader(

    test_dataset,

    batch_size=4,

    shuffle=False

)



print("="*70)

print("TEST DATASET CREATED")

print("="*70)


print(
    "Samples:",
    len(test_dataset)
)


print(
    "Batches:",
    len(test_loader)
)


# Check one batch

batch = next(iter(test_loader))


print()

print("Clinical:",
      batch["clinical"].shape)


print("Genomic:",
      batch["genomic"].shape)


print("Pathology:",
      batch["pathology"].shape)


print("Events:",
      batch["event"].shape)

TEST DATASET CREATED
Samples: 54
Batches: 14

Clinical: torch.Size([4, 20])
Genomic: torch.Size([4, 200])
Pathology: torch.Size([4, 512, 512])
Events: torch.Size([4])



##Define Transformer Architecture

In [10]:
# ============================================================
# TRANSFORMER ARCHITECTURE
# ============================================================


import torch
import torch.nn as nn



# ============================================================
# CLINICAL ENCODER
# ============================================================

class ClinicalEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embed_dim,
        dropout
    ):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                embed_dim
            ),

            nn.LayerNorm(
                embed_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )
        )


    def forward(self,x):

        return self.encoder(x).unsqueeze(1)



# ============================================================
# GENOMIC ENCODER
# ============================================================

# ============================================================
# GENOMIC ENCODER (MATCH TRAINED MODEL)
# ============================================================

class GenomicEncoder(nn.Module):


    def __init__(
        self,
        input_dim,
        embed_dim,
        dropout
    ):

        super().__init__()


        self.encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.LayerNorm(
                256
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),


            nn.Linear(
                256,
                embed_dim
            ),

            nn.LayerNorm(
                embed_dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            )

        )



    def forward(self,x):

        return self.encoder(x).unsqueeze(1)




# ============================================================
# PATHOLOGY ENCODER
# ============================================================

class PathologyEncoder(nn.Module):


    def __init__(
        self,
        input_dim,
        embed_dim,
        num_heads,
        num_layers,
        dropout
    ):

        super().__init__()


        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                embed_dim
            )
        )


        self.patch_projection = nn.Linear(
            input_dim,
            embed_dim
        )


        encoder_layer = nn.TransformerEncoderLayer(

            d_model=embed_dim,

            nhead=num_heads,

            dim_feedforward=embed_dim*4,

            dropout=dropout,

            batch_first=True,

            activation="gelu"

        )


        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=num_layers

        )


        self.norm = nn.LayerNorm(
            embed_dim
        )



    def forward(
        self,
        x,
        mask=None
    ):


        batch = x.size(0)


        x = self.patch_projection(x)


        cls = self.cls_token.expand(
            batch,
            -1,
            -1
        )


        x = torch.cat(
            [
                cls,
                x
            ],
            dim=1
        )


        x = self.transformer(x)


        x = self.norm(x[:,0])


        return x.unsqueeze(1)





# ============================================================
# FUSION TRANSFORMER
# ============================================================

class MultimodalFusionTransformer(nn.Module):


    def __init__(
        self,
        embed_dim,
        num_heads,
        num_layers,
        dropout
    ):

        super().__init__()


        layer = nn.TransformerEncoderLayer(

            d_model=embed_dim,

            nhead=num_heads,

            dim_feedforward=embed_dim*4,

            dropout=dropout,

            batch_first=True,

            activation="gelu"

        )


        self.transformer = nn.TransformerEncoder(

            layer,

            num_layers=num_layers

        )


        self.norm = nn.LayerNorm(
            embed_dim
        )



    def forward(
        self,
        clinical_token,
        genomic_token,
        pathology_token
    ):


        x = torch.cat(

            [
                clinical_token,
                genomic_token,
                pathology_token
            ],

            dim=1
        )


        x = self.transformer(x)


        x = self.norm(x)


        return x.mean(dim=1)




# ============================================================
# SURVIVAL HEAD
# ============================================================

class SurvivalPredictionHead(nn.Module):


    def __init__(
        self,
        embed_dim,
        dropout
    ):

        super().__init__()


        self.head = nn.Sequential(

            nn.Linear(
                embed_dim,
                64
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                64,
                1
            )

        )


    def forward(self,x):

        return self.head(x)




# ============================================================
# COMPLETE MODEL
# ============================================================

class MultimodalTransformer(nn.Module):


    def __init__(
        self,
        config
    ):

        super().__init__()


        self.clinical_encoder = ClinicalEncoder(
            config["clinical_dim"],
            config["embed_dim"],
            config["dropout"]
        )


        self.genomic_encoder = GenomicEncoder(
            config["genomic_dim"],
            config["embed_dim"],
            config["dropout"]
        )


        self.pathology_encoder = PathologyEncoder(
            config["pathology_dim"],
            config["embed_dim"],
            config["num_heads"],
            config["num_layers"],
            config["dropout"]
        )


        self.fusion = MultimodalFusionTransformer(
            config["embed_dim"],
            config["num_heads"],
            config["num_layers"],
            config["dropout"]
        )


        self.survival_head = SurvivalPredictionHead(
            config["embed_dim"],
            config["dropout"]
        )



    def forward(
        self,
        clinical,
        genomic,
        pathology,
        mask=None
    ):


        c = self.clinical_encoder(
            clinical
        )


        g = self.genomic_encoder(
            genomic
        )


        p = self.pathology_encoder(
            pathology,
            mask
        )


        fused = self.fusion(
            c,
            g,
            p
        )


        risk = self.survival_head(
            fused
        )


        return risk.squeeze(1)


##Create Model and Load trained checkpoint

In [11]:
# ============================================================
# CREATE MODEL + LOAD CHECKPOINT
# ============================================================


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


MODEL_CONFIG = {

    "clinical_dim": clinical_dim,

    "genomic_dim": genomic_dim,

    "pathology_dim": 512,

    "embed_dim": 128,

    "num_heads": 4,

    "num_layers": 2,

    "dropout": 0.3

}


print("="*70)
print("MODEL CONFIGURATION")
print("="*70)

print(MODEL_CONFIG)



model = MultimodalTransformer(
    MODEL_CONFIG
)


model = model.to(device)



checkpoint = torch.load(

    CHECKPOINT_PATH,

    map_location=device

)


model.load_state_dict(
    checkpoint
)


model.eval()


print("="*70)
print("TRANSFORMER LOADED SUCCESSFULLY")
print("="*70)

MODEL CONFIGURATION
{'clinical_dim': 20, 'genomic_dim': 200, 'pathology_dim': 512, 'embed_dim': 128, 'num_heads': 4, 'num_layers': 2, 'dropout': 0.3}
TRANSFORMER LOADED SUCCESSFULLY



##Run Transformer Inference

In [12]:
# ============================================================
# TRANSFORMER TEST PREDICTION
# ============================================================


transformer_risk = []


model.eval()


with torch.no_grad():

    for batch in test_loader:


        clinical = batch["clinical"].to(device)

        genomic = batch["genomic"].to(device)

        pathology = batch["pathology"].to(device)

        mask = batch["mask"].to(device)



        output = model(

            clinical,

            genomic,

            pathology,

            mask

        )


        # model returns dictionary
        # Handle both model output formats

        if isinstance(output, dict):

           risk = output["survival"]

        else:

           risk = output


        transformer_risk.extend(

            risk.squeeze()
            .cpu()
            .numpy()
            .tolist()

        )



transformer_risk = np.array(
    transformer_risk
)



# Use the same test dataframe outcomes
transformer_time = test_df["survival_time"].values

transformer_event = test_df["event"].astype(bool).values



print("="*70)
print("TRANSFORMER PREDICTION COMPLETE")
print("="*70)


print(
    "Risk shape:",
    transformer_risk.shape
)


print(
    "Survival shape:",
    transformer_time.shape
)


print(
    "Event shape:",
    transformer_event.shape
)

TRANSFORMER PREDICTION COMPLETE
Risk shape: (54,)
Survival shape: (54,)
Event shape: (54,)


##Calculate Transformer C-index

In [13]:
# ============================================================
# TRANSFORMER C-INDEX
# ============================================================


from lifelines.utils import concordance_index


transformer_cindex = concordance_index(

    transformer_time,

    -transformer_risk,

    transformer_event

)


print("="*70)
print("TRANSFORMER PERFORMANCE")
print("="*70)


print(
    "Transformer C-index:",
    transformer_cindex
)

TRANSFORMER PERFORMANCE
Transformer C-index: 0.8227848101265823


##Cox models

In [14]:
# ============================================================
# LOAD TRAIN AND TEST DATA FOR COX / RSF
# ============================================================

RFS_DIR = "/content/drive/MyDrive/TCGA_BRCA/rfs_feature_selection"


train_df = pd.read_csv(
    os.path.join(
        RFS_DIR,
        "rfs_train_selected.csv"
    )
)


test_df = pd.read_csv(
    os.path.join(
        RFS_DIR,
        "rfs_test_selected.csv"
    )
)


print("="*70)
print("COX / RSF DATA LOADED")
print("="*70)

print("Train:", train_df.shape)
print("Test :", test_df.shape)


display(train_df.head())
display(test_df.head())

COX / RSF DATA LOADED
Train: (249, 224)
Test : (54, 224)


,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,histological_type_infiltratinglobularcarcinoma,histological_type_medullarycarcinoma,...,HAPLN1,RNF183,LOC96610,TMPRSS4,C6orf141,C20orf85,patient_id,file_uuid,event,survival_time
0,-0.809452,-0.491222,1,1,0,0,-0.502090,1,False,False,...,-0.566671,-0.634398,1.293934,-0.556140,-0.189218,-0.514648,TCGA-A2-A0SX,64093c9e-aebe-4367-adbb-3ea85e3234c9,1,1534.0
1,-1.802022,0.577139,2,2,1,0,0.197181,1,False,False,...,-0.106182,1.473455,0.459915,-1.182343,-0.105079,-0.514648,TCGA-A2-A3XU,87e7cf96-5ab5-4f26-b839-25cd31607c9c,1,912.0
2,0.030415,-0.399479,3,2,2,0,-0.269000,0,False,False,...,0.419087,0.056340,0.475637,-0.237924,-0.540717,1.622850,TCGA-C8-A1HI,1bf2c09e-854f-414f-9b5e-2ad8a5176abd,0,343.0
3,-0.885803,-0.302557,4,2,1,1,1.362635,1,False,False,...,-1.230714,0.020616,0.521725,-1.010735,0.551956,-0.328043,TCGA-BH-A1FH,dd988de8-397e-488d-aa6b-03de44276da4,0,1034.0
4,-1.420264,-1.325047,3,3,1,0,1.595725,0,False,False,...,-0.144315,-0.650541,0.660375,-0.181828,-1.540601,-0.514648,TCGA-B6-A0WX,8fef8f57-cb43-43d7-a442-97d095ad203e,1,639.0


,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,histological_type_infiltratinglobularcarcinoma,histological_type_medullarycarcinoma,...,HAPLN1,RNF183,LOC96610,TMPRSS4,C6orf141,C20orf85,patient_id,file_uuid,event,survival_time
0,0.717579,0.130262,3,2,3,0,1.828816,1,True,False,...,-1.306939,-0.512511,-0.574521,0.916788,0.432721,-0.514648,TCGA-AR-A5QN,88e15db1-54ef-43e8-91d6-bb41d9fb3cf1,0,1013.0
1,0.259470,-0.329932,2,2,0,0,-0.502090,1,False,False,...,0.247674,0.088457,-0.658618,0.245905,-0.809119,-0.514648,TCGA-BH-A0DE,0bd7ca22-a281-427c-a5ef-b59b5dd02da3,0,2372.0
2,-1.572967,0.625970,3,3,1,0,0.896453,1,False,False,...,-0.174433,0.258773,1.633520,0.803308,1.029709,-0.313943,TCGA-B6-A0IE,20d69f17-d957-4dc2-ae29-3f28a9518868,1,1993.0
3,-0.962155,-0.614039,2,2,1,0,-0.035909,1,False,False,...,-0.024339,0.841921,0.888047,-0.482192,1.047505,-0.514648,TCGA-GM-A2DA,ac8c09f2-2fd6-43e2-8233-00dcf54dcb0b,1,6593.0
4,-1.496616,1.149053,2,2,1,0,0.197181,1,False,False,...,-1.167379,-0.050998,0.512568,1.291236,1.465347,-0.300371,TCGA-OL-A66O,1c8bf463-54da-4f25-a146-e9ecb3df3af1,0,528.0


In [15]:
# ============================================================
# COX MODEL EVALUATION
# ============================================================

from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler


def run_cox_model(
    name,
    features

):

    print("="*70)
    print(f"Running Cox: {name}")
    print("="*70)


    train_cox = train_df[
        features +
        [
            "survival_time",
            "event"
        ]
    ].copy()


    test_cox = test_df[
        features +
        [
            "survival_time",
            "event"
        ]
    ].copy()



    # remove constant columns

    variance = train_cox[features].var()

    valid_features = variance[
        variance > 1e-8
    ].index.tolist()



    print(
        "Removed constant features:",
        len(features)-len(valid_features)
    )



    train_cox = train_cox[
        valid_features +
        [
            "survival_time",
            "event"
        ]
    ]



    test_cox = test_cox[
        valid_features +
        [
            "survival_time",
            "event"
        ]
    ]



    # scale features

    scaler = StandardScaler()


    train_cox[valid_features] = scaler.fit_transform(
        train_cox[valid_features]
    )


    test_cox[valid_features] = scaler.transform(
        test_cox[valid_features]
    )



    # Cox

    cph = CoxPHFitter(
        penalizer=0.1
    )


    cph.fit(
        train_cox,
        duration_col="survival_time",
        event_col="event"
    )


    risk = cph.predict_partial_hazard(
        test_cox
    ).values



    c_index = concordance_index_censored(
        test_cox["event"].astype(bool),
        test_cox["survival_time"],
        risk
    )[0]



    print(
        "C-index:",
        c_index
    )


    return c_index

In [16]:
# ============================================================
# RUN COX MODELS
# ============================================================


cox_results = {}


# ----------------------------
# Clinical
# ----------------------------

cox_results["Clinical"] = run_cox_model(
    "Clinical",
    clinical_features
)



# ----------------------------
# Genomic
# ----------------------------

cox_results["Genomic"] = run_cox_model(
    "Genomic",
    genomic_features
)



# ----------------------------
# Clinical + Genomic
# ----------------------------

clinical_genomic_features = (
    clinical_features +
    genomic_features
)


cox_results["Clinical_Genomic"] = run_cox_model(
    "Clinical + Genomic",
    clinical_genomic_features
)



print("="*70)
print("COX RESULTS")
print("="*70)

print(cox_results)

Running Cox: Clinical
Removed constant features: 0
C-index: 0.7531645569620253
Running Cox: Genomic
Removed constant features: 0
C-index: 0.6962025316455697
Running Cox: Clinical + Genomic
Removed constant features: 0
C-index: 0.7056962025316456
COX RESULTS
{'Clinical': np.float64(0.7531645569620253), 'Genomic': np.float64(0.6962025316455697), 'Clinical_Genomic': np.float64(0.7056962025316456)}


##RSF baseline

In [17]:
# ============================================================
# RSF MODEL EVALUATION
# ============================================================


from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sklearn.preprocessing import StandardScaler



def run_rsf_model(
    name,
    features
):

    print("="*70)
    print(f"Running RSF: {name}")
    print("="*70)



    X_train = train_df[
        features
    ].copy()


    X_test = test_df[
        features
    ].copy()



    y_train = Surv.from_arrays(

        train_df["event"].astype(bool),

        train_df["survival_time"]

    )



    y_test = Surv.from_arrays(

        test_df["event"].astype(bool),

        test_df["survival_time"]

    )



    # remove constant features

    variance = X_train.var()


    valid_features = variance[
        variance > 1e-8
    ].index.tolist()



    print(
        "Removed constant features:",
        len(features)-len(valid_features)
    )



    X_train = X_train[
        valid_features
    ]


    X_test = X_test[
        valid_features
    ]



    # scaling

    scaler = StandardScaler()


    X_train = scaler.fit_transform(
        X_train
    )


    X_test = scaler.transform(
        X_test
    )



    model = RandomSurvivalForest(

        n_estimators=300,

        min_samples_split=10,

        min_samples_leaf=5,

        max_features="sqrt",

        random_state=42,

        n_jobs=-1

    )



    model.fit(

        X_train,

        y_train

    )



    risk = model.predict(
        X_test
    )



    c_index = concordance_index_censored(

        y_test["event"],

        y_test["time"],

        risk

    )[0]



    print(
        "C-index:",
        c_index
    )


    return c_index

In [18]:
# ============================================================
# RUN RSF MODELS
# ============================================================


rsf_results = {}


rsf_results["Clinical"] = run_rsf_model(
    "Clinical",
    clinical_features
)


rsf_results["Genomic"] = run_rsf_model(
    "Genomic",
    genomic_features
)


rsf_results["Clinical_Genomic"] = run_rsf_model(
    "Clinical + Genomic",
    clinical_features + genomic_features
)



print("="*70)
print("RSF RESULTS")
print("="*70)

print(rsf_results)

Running RSF: Clinical
Removed constant features: 0
C-index: 0.7436708860759493
Running RSF: Genomic
Removed constant features: 0
C-index: 0.6487341772151899
Running RSF: Clinical + Genomic
Removed constant features: 0
C-index: 0.6550632911392406
RSF RESULTS
{'Clinical': np.float64(0.7436708860759493), 'Genomic': np.float64(0.6487341772151899), 'Clinical_Genomic': np.float64(0.6550632911392406)}


##Comparsion Table

In [19]:
# ============================================================
# SAVE TRANSFORMER RESULT
# ============================================================

transformer_c_index = transformer_cindex

print(
    "Transformer C-index:",
    transformer_c_index
)

Transformer C-index: 0.8227848101265823


In [20]:
# ============================================================
# FINAL MODEL COMPARISON TABLE
# ============================================================


comparison_results = pd.DataFrame([

    {
        "Model": "CoxPH",
        "Modalities": "Clinical",
        "C-index": cox_results["Clinical"]
    },

    {
        "Model": "CoxPH",
        "Modalities": "Genomic",
        "C-index": cox_results["Genomic"]
    },

    {
        "Model": "CoxPH",
        "Modalities": "Clinical + Genomic",
        "C-index": cox_results["Clinical_Genomic"]
    },


    {
        "Model": "RSF",
        "Modalities": "Clinical",
        "C-index": rsf_results["Clinical"]
    },

    {
        "Model": "RSF",
        "Modalities": "Genomic",
        "C-index": rsf_results["Genomic"]
    },

    {
        "Model": "RSF",
        "Modalities": "Clinical + Genomic",
        "C-index": rsf_results["Clinical_Genomic"]
    },


    {
        "Model": "Multimodal Transformer",
        "Modalities": "Clinical + Genomic + Pathology",
        "C-index": transformer_c_index
    }

])


comparison_results = comparison_results.sort_values(
    by="C-index",
    ascending=False
).reset_index(drop=True)



print("="*70)
print("FINAL MODEL COMPARISON")
print("="*70)


display(comparison_results)

FINAL MODEL COMPARISON


,Model,Modalities,C-index
0,Multimodal Transformer,Clinical + Genomic + Pathology,0.822785
1,CoxPH,Clinical,0.753165
2,RSF,Clinical,0.743671
3,CoxPH,Clinical + Genomic,0.705696
4,CoxPH,Genomic,0.696203
5,RSF,Clinical + Genomic,0.655063
6,RSF,Genomic,0.648734


##Load pathology features

In [21]:
# ============================================================
# CREATE PATIENT-LEVEL PATHOLOGY FEATURES
# ============================================================

import glob
import os
import numpy as np
import pandas as pd

PATHOLOGY_DIR = "/content/drive/MyDrive/TCGA_BRCA/Features"

feature_files = sorted(
    glob.glob(
        os.path.join(PATHOLOGY_DIR, "*.npy")
    )
)

print("="*70)
print("PATHOLOGY FILES")
print("="*70)
print("Total patients:", len(feature_files))

PATHOLOGY FILES
Total patients: 372


##Mean Pooling

In [23]:
# ============================================================
# BUILD PATHOLOGY FEATURES ONLY FOR TRAIN/TEST PATIENTS
# ============================================================

def extract_patient_pathology(df):

    rows = []

    for _, row in df.iterrows():

        uuid = row["file_uuid"]

        file = os.path.join(
            FEATURE_DIR,
            uuid + ".npy"
        )

        patches = np.load(file)

        pooled = patches.mean(axis=0)

        patient = {
            "file_uuid": uuid
        }

        for i in range(512):
            patient[f"pathology_{i}"] = pooled[i]

        rows.append(patient)

    return pd.DataFrame(rows)


print("="*70)
print("Building TRAIN pathology features")
print("="*70)

train_pathology = extract_patient_pathology(train_df)

print(train_pathology.shape)


print("="*70)
print("Building TEST pathology features")
print("="*70)

test_pathology = extract_patient_pathology(test_df)

print(test_pathology.shape)

Building TRAIN pathology features
(249, 513)
Building TEST pathology features
(54, 513)


In [24]:
print(train_df.shape)
print(test_df.shape)

(249, 224)
(54, 224)


##Merge

In [25]:
# ============================================================
# MERGE PATHOLOGY WITH TRAIN / TEST
# ============================================================

train_path = train_df.merge(
    train_pathology,
    on="file_uuid",
    how="inner"
)

test_path = test_df.merge(
    test_pathology,
    on="file_uuid",
    how="inner"
)

print("="*70)
print("AFTER MERGING")
print("="*70)

print("Train:", train_path.shape)
print("Test :", test_path.shape)

AFTER MERGING
Train: (249, 736)
Test : (54, 736)



##Define Pathology Features

In [26]:
# ============================================================
# DEFINE PATHOLOGY FEATURES
# ============================================================

pathology_features = [

    c for c in train_path.columns

    if c.startswith("pathology_")

]


clinical_genomic_pathology_features = (

    clinical_features

    +

    genomic_features

    +

    pathology_features

)


print("="*70)

print("Pathology features:", len(pathology_features))

print("Total features:",
      len(clinical_genomic_pathology_features))

print("="*70)

Pathology features: 515
Total features: 735


##Cox with pathology

In [27]:
# ============================================================
# COX MODEL EVALUATION (GENERAL VERSION)
# ============================================================

from lifelines import CoxPHFitter
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_censored


def run_cox_model(
    name,
    features,
    train_dataframe,
    test_dataframe
):

    print("="*70)
    print(f"Running Cox: {name}")
    print("="*70)


    train_cox = train_dataframe[
        features +
        [
            "survival_time",
            "event"
        ]
    ].copy()


    test_cox = test_dataframe[
        features +
        [
            "survival_time",
            "event"
        ]
    ].copy()



    # remove constant features

    variance = train_cox[features].var()

    valid_features = variance[
        variance > 1e-8
    ].index.tolist()


    print(
        "Removed constant features:",
        len(features)-len(valid_features)
    )


    train_cox = train_cox[
        valid_features +
        [
            "survival_time",
            "event"
        ]
    ]


    test_cox = test_cox[
        valid_features +
        [
            "survival_time",
            "event"
        ]
    ]


    # scaling

    scaler = StandardScaler()


    train_cox[valid_features] = scaler.fit_transform(
        train_cox[valid_features]
    )


    test_cox[valid_features] = scaler.transform(
        test_cox[valid_features]
    )


    # Cox model

    cph = CoxPHFitter(
        penalizer=0.1
    )


    cph.fit(
        train_cox,
        duration_col="survival_time",
        event_col="event"
    )


    risk = cph.predict_partial_hazard(
        test_cox
    ).values



    c_index = concordance_index_censored(
        test_cox["event"].astype(bool),
        test_cox["survival_time"],
        risk
    )[0]


    print(
        "C-index:",
        c_index
    )


    return c_index

In [28]:
# ============================================================
# RUN COX WITH CLINICAL + GENOMIC + PATHOLOGY
# ============================================================

cox_pathology = run_cox_model(

    "Clinical + Genomic + Pathology",

    clinical_genomic_pathology_features,

    train_dataframe=train_path,

    test_dataframe=test_path

)


print("="*70)
print("COX PATHOLOGY RESULT")
print("="*70)

print(cox_pathology)

Running Cox: Clinical + Genomic + Pathology
Removed constant features: -6
C-index: 0.6645569620253164
COX PATHOLOGY RESULT
0.6645569620253164


##RSF

In [29]:
# ============================================================
# Clinical + Genomic + Pathology support
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored


def run_rsf_model(
    name,
    features,
    train_dataframe,
    test_dataframe
):

    print("="*70)
    print(f"Running RSF: {name}")
    print("="*70)


    # -----------------------------
    # Prepare train/test
    # -----------------------------

    X_train = train_dataframe[
        features
    ].copy()


    X_test = test_dataframe[
        features
    ].copy()


    y_train = Surv.from_dataframe(
        "event",
        "survival_time",
        train_dataframe
    )


    y_test_event = test_dataframe[
        "event"
    ].astype(bool)


    y_test_time = test_dataframe[
        "survival_time"
    ]



    # -----------------------------
    # Remove constant features
    # -----------------------------

    variance = X_train.var()


    valid_features = variance[
        variance > 1e-8
    ].index.tolist()



    print(
        "Removed constant features:",
        len(features)-len(valid_features)
    )


    X_train = X_train[
        valid_features
    ]

    X_test = X_test[
        valid_features
    ]



    # -----------------------------
    # Scaling
    # -----------------------------

    scaler = StandardScaler()


    X_train = scaler.fit_transform(
        X_train
    )


    X_test = scaler.transform(
        X_test
    )



    # -----------------------------
    # RSF
    # -----------------------------

    rsf = RandomSurvivalForest(

        n_estimators=300,

        min_samples_split=10,

        min_samples_leaf=5,

        max_features="sqrt",

        n_jobs=-1,

        random_state=42

    )



    rsf.fit(
        X_train,
        y_train
    )



    # -----------------------------
    # Prediction
    # -----------------------------

    risk = -rsf.predict(
        X_test
    )



    # -----------------------------
    # C-index
    # -----------------------------

    c_index = concordance_index_censored(

        y_test_event,

        y_test_time,

        risk

    )[0]


    print(
        "C-index:",
        c_index
    )


    return c_index

In [30]:
# ============================================================
# RUN RSF WITH CLINICAL + GENOMIC + PATHOLOGY
# ============================================================


rsf_pathology = run_rsf_model(

    "Clinical + Genomic + Pathology",

    clinical_genomic_pathology_features,

    train_dataframe=train_path,

    test_dataframe=test_path

)


print("="*70)
print("RSF PATHOLOGY RESULT")
print("="*70)

print(rsf_pathology)

Running RSF: Clinical + Genomic + Pathology
Removed constant features: 0
C-index: 0.26582278481012656
RSF PATHOLOGY RESULT
0.26582278481012656


##Update comparison table

In [31]:
# ============================================================
# UPDATED COMPARISON
# ============================================================

comparison = pd.DataFrame([

    {
        "Model":"Multimodal Transformer",
        "Modalities":"Clinical + Genomic + Pathology",
        "C-index":transformer_cindex
    },

    {
        "Model":"CoxPH",
        "Modalities":"Clinical",
        "C-index":cox_results["Clinical"]
    },

    {
        "Model":"RSF",
        "Modalities":"Clinical",
        "C-index":rsf_results["Clinical"]
    },

    {
        "Model":"CoxPH",
        "Modalities":"Clinical + Genomic",
        "C-index":cox_results["Clinical_Genomic"]
    },

    {
        "Model":"RSF",
        "Modalities":"Clinical + Genomic",
        "C-index":rsf_results["Clinical_Genomic"]
    },

    {
        "Model":"CoxPH",
        "Modalities":"Clinical + Genomic + Pathology",
        "C-index":cox_pathology
    },

    {
        "Model":"RSF",
        "Modalities":"Clinical + Genomic + Pathology",
        "C-index":rsf_pathology
    }

])

comparison = comparison.sort_values(
    "C-index",
    ascending=False
).reset_index(drop=True)

display(comparison)

,Model,Modalities,C-index
0,Multimodal Transformer,Clinical + Genomic + Pathology,0.822785
1,CoxPH,Clinical,0.753165
2,RSF,Clinical,0.743671
3,CoxPH,Clinical + Genomic,0.705696
4,CoxPH,Clinical + Genomic + Pathology,0.664557
5,RSF,Clinical + Genomic,0.655063
6,RSF,Clinical + Genomic + Pathology,0.265823


##Prepare PCA pathology features

In [32]:
# ============================================================
# PCA REDUCTION FOR PATHOLOGY FEATURES
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


print("="*70)
print("PCA PATHOLOGY REDUCTION")
print("="*70)


# pathology columns
pathology_columns = [
    c for c in train_path.columns
    if c.startswith("pathology_")
]


print(
    "Original pathology features:",
    len(pathology_columns)
)


# Standardize pathology embeddings

path_scaler = StandardScaler()


train_path_scaled = path_scaler.fit_transform(
    train_path[pathology_columns]
)


test_path_scaled = path_scaler.transform(
    test_path[pathology_columns]
)



# PCA

pca = PCA(
    n_components=64,
    random_state=42
)


train_path_pca = pca.fit_transform(
    train_path_scaled
)


test_path_pca = pca.transform(
    test_path_scaled
)



print(
    "After PCA:",
    train_path_pca.shape,
    test_path_pca.shape
)


print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)

PCA PATHOLOGY REDUCTION
Original pathology features: 515
After PCA: (249, 64) (54, 64)
Explained variance: 0.9745451792620371


##Create PCA pathology dataframe

In [33]:
# ============================================================
# CREATE PCA PATHOLOGY DATAFRAME
# ============================================================


pca_columns = [
    f"pathology_pca_{i}"
    for i in range(64)
]


train_pca_df = pd.DataFrame(
    train_path_pca,
    columns=pca_columns
)


test_pca_df = pd.DataFrame(
    test_path_pca,
    columns=pca_columns
)



# keep patient identifiers

train_pca_df["file_uuid"] = train_path["file_uuid"].values

test_pca_df["file_uuid"] = test_path["file_uuid"].values



print(train_pca_df.shape)
print(test_pca_df.shape)


display(train_pca_df.head())

(249, 65)
(54, 65)


,pathology_pca_0,pathology_pca_1,pathology_pca_2,pathology_pca_3,pathology_pca_4,pathology_pca_5,pathology_pca_6,pathology_pca_7,pathology_pca_8,pathology_pca_9,...,pathology_pca_55,pathology_pca_56,pathology_pca_57,pathology_pca_58,pathology_pca_59,pathology_pca_60,pathology_pca_61,pathology_pca_62,pathology_pca_63,file_uuid
0,9.289393,-9.049829,-0.737777,-3.368697,3.206657,-0.314733,10.122093,-0.966973,1.060365,3.309567,...,-0.644720,-0.534953,-0.237316,-0.282747,-0.935565,1.059695,0.320457,-0.304430,0.047669,64093c9e-aebe-4367-adbb-3ea85e3234c9
1,-12.638244,-8.529000,-5.194415,-4.452182,-4.333612,6.703958,5.860002,1.981025,-0.905466,2.396708,...,-0.787202,0.178480,1.325069,-0.317790,0.281234,0.544642,1.413503,-0.730911,0.892010,87e7cf96-5ab5-4f26-b839-25cd31607c9c
2,0.471626,5.147265,-6.437668,-1.925770,-1.238297,5.255552,-3.055885,1.776464,1.426411,-0.767186,...,-0.913318,-0.099976,-0.483488,-0.122317,-0.277295,-0.034871,0.225359,0.981302,0.025354,1bf2c09e-854f-414f-9b5e-2ad8a5176abd
3,7.544954,-12.362501,-6.057413,-3.705898,-4.286909,-9.014600,-1.543155,3.700825,-0.139246,-0.965504,...,-0.826522,-0.157261,-0.035791,-0.290577,0.561254,-0.413918,-0.390871,0.468904,-0.302759,dd988de8-397e-488d-aa6b-03de44276da4
4,-7.902311,-13.918295,2.666240,-10.448858,-5.929704,-9.415015,-3.939012,-4.280120,-4.503107,2.158997,...,1.026913,0.710976,0.300043,-1.336618,-0.091044,0.798085,0.152131,-0.739154,-0.685445,8fef8f57-cb43-43d7-a442-97d095ad203e


##Merge PCA pathology with clinical + genomic

In [34]:
# ============================================================
# MERGE PCA PATHOLOGY FEATURES
# ============================================================


train_pca_full = train_df.merge(
    train_pca_df,
    on="file_uuid",
    how="inner"
)


test_pca_full = test_df.merge(
    test_pca_df,
    on="file_uuid",
    how="inner"
)



print("="*70)
print("PCA DATASET")
print("="*70)


print(
    "Train:",
    train_pca_full.shape
)

print(
    "Test:",
    test_pca_full.shape
)

PCA DATASET
Train: (249, 288)
Test: (54, 288)


##Define PCA pathology feature list

In [35]:
# ============================================================
# PCA FEATURE LIST
# ============================================================


pca_pathology_features = [
    c for c in train_pca_full.columns
    if c.startswith("pathology_pca_")
]


clinical_genomic_pca_features = (
    clinical_features
    +
    genomic_features
    +
    pca_pathology_features
)



print(
    "Clinical:",
    len(clinical_features)
)

print(
    "Genomic:",
    len(genomic_features)
)

print(
    "PCA Pathology:",
    len(pca_pathology_features)
)

print(
    "Total:",
    len(clinical_genomic_pca_features)
)

Clinical: 20
Genomic: 200
PCA Pathology: 64
Total: 284


##Run RSF with PCA pathology

In [36]:
# ============================================================
# RSF WITH PCA PATHOLOGY
# ============================================================


rsf_pca_pathology = run_rsf_model(
    "Clinical + Genomic + PCA Pathology",
    clinical_genomic_pca_features,
    train_pca_full,
    test_pca_full
)


print("="*70)
print("RSF PCA PATHOLOGY RESULT")
print("="*70)

print(
    rsf_pca_pathology
)

Running RSF: Clinical + Genomic + PCA Pathology
Removed constant features: 0
C-index: 0.3259493670886076
RSF PCA PATHOLOGY RESULT
0.3259493670886076


##Add to comparison table

In [37]:
# ============================================================
# UPDATE COMPARISON TABLE WITH PCA PATHOLOGY RSF
# ============================================================


new_result = pd.DataFrame([

    {
        "Model":
        "RSF",

        "Modalities":
        "Clinical + Genomic + PCA Pathology",

        "C-index":
        rsf_pca_pathology
    }

])



comparison = pd.concat(
    [
        comparison,
        new_result
    ],
    ignore_index=True
)



comparison = comparison.sort_values(
    "C-index",
    ascending=False
).reset_index(drop=True)



display(comparison)

,Model,Modalities,C-index
0,Multimodal Transformer,Clinical + Genomic + Pathology,0.822785
1,CoxPH,Clinical,0.753165
2,RSF,Clinical,0.743671
3,CoxPH,Clinical + Genomic,0.705696
4,CoxPH,Clinical + Genomic + Pathology,0.664557
5,RSF,Clinical + Genomic,0.655063
6,RSF,Clinical + Genomic + PCA Pathology,0.325949
7,RSF,Clinical + Genomic + Pathology,0.265823


##CoxPH + PCA Pathology

In [38]:
cox_pca_pathology = run_cox_model(
    "Clinical + Genomic + PCA Pathology",
    clinical_features
    + genomic_features
    + pca_pathology_features,
    train_pca_full,
    test_pca_full
)

print("="*70)
print("COX PCA PATHOLOGY RESULT")
print("="*70)

print(cox_pca_pathology)

Running Cox: Clinical + Genomic + PCA Pathology
Removed constant features: 0
C-index: 0.6613924050632911
COX PCA PATHOLOGY RESULT
0.6613924050632911


##RSF + PCA pathology only

In [39]:
rsf_pathology_only = run_rsf_model(
    "PCA Pathology Only",
    pca_pathology_features,
    train_pca_full,
    test_pca_full
)

print(rsf_pathology_only)

Running RSF: PCA Pathology Only
Removed constant features: 0
C-index: 0.5316455696202531
0.5316455696202531


##CoxPH + Pathology only

In [40]:
cox_pathology_only = run_cox_model(
    "Pathology Only",
    pca_pathology_features,
    train_pca_full,
    test_pca_full
)

print(cox_pathology_only)

Running Cox: Pathology Only
Removed constant features: 0
C-index: 0.5822784810126582
0.5822784810126582


##Last comparsion table

In [41]:
# ============================================================
# ADD PATHOLOGY ONLY RESULTS TO COMPARISON TABLE
# ============================================================


new_results = pd.DataFrame([

    {
        "Model": "RSF",
        "Modalities": "PCA Pathology Only",
        "C-index": rsf_pathology_only
    },

    {
        "Model": "CoxPH",
        "Modalities": "PCA Pathology Only",
        "C-index": cox_pathology_only
    }

])


comparison = pd.concat(
    [
        comparison,
        new_results
    ],
    ignore_index=True
)


comparison = comparison.sort_values(
    "C-index",
    ascending=False
).reset_index(drop=True)


display(comparison)

,Model,Modalities,C-index
0,Multimodal Transformer,Clinical + Genomic + Pathology,0.822785
1,CoxPH,Clinical,0.753165
2,RSF,Clinical,0.743671
3,CoxPH,Clinical + Genomic,0.705696
4,CoxPH,Clinical + Genomic + Pathology,0.664557
5,RSF,Clinical + Genomic,0.655063
6,CoxPH,PCA Pathology Only,0.582278
7,RSF,PCA Pathology Only,0.531646
8,RSF,Clinical + Genomic + PCA Pathology,0.325949
9,RSF,Clinical + Genomic + Pathology,0.265823


##Prepare pathology-only data

In [56]:
# ============================================================
# PATHOLOGY IMAGE EMBEDDING FEATURES ONLY
# ============================================================


pathology_columns = [

    c for c in train_path.columns

    if c.startswith("pathology_")

    and c.replace(
        "pathology_",
        ""
    ).isdigit()

]


print("="*70)
print("PATHOLOGY IMAGE FEATURES")
print("="*70)


print(
    "Number of pathology features:",
    len(pathology_columns)
)


print(
    pathology_columns[:5]
)

print(
    pathology_columns[-5:]
)

PATHOLOGY IMAGE FEATURES
Number of pathology features: 512
['pathology_0', 'pathology_1', 'pathology_2', 'pathology_3', 'pathology_4']
['pathology_507', 'pathology_508', 'pathology_509', 'pathology_510', 'pathology_511']


##Dataset

In [57]:
# ============================================================
# PATHOLOGY DATASET
# ============================================================


class PathologySurvivalDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.data = dataframe.reset_index(drop=True)


    def __len__(self):

        return len(self.data)


    def __getitem__(
        self,
        idx
    ):

        row = self.data.iloc[idx]


        x = row[pathology_columns].values.astype(
            np.float32
        )


        event = np.float32(
            row["event"]
        )


        survival = np.float32(
            row["survival_time"]
        )


        return {

            "pathology":
            torch.tensor(x),


            "event":
            torch.tensor(event),


            "survival":
            torch.tensor(survival)

        }

##Create datasets and loaders

In [58]:
# ============================================================
# PATHOLOGY LOADERS
# ============================================================


path_train_dataset = PathologySurvivalDataset(
    train_path
)


path_test_dataset = PathologySurvivalDataset(
    test_path
)



path_train_loader = DataLoader(
    path_train_dataset,
    batch_size=16,
    shuffle=True
)


path_test_loader = DataLoader(
    path_test_dataset,
    batch_size=16,
    shuffle=False
)


print("="*70)
print("PATHOLOGY LOADERS CREATED")
print("="*70)


print(
    "Train:",
    len(path_train_dataset)
)


print(
    "Test:",
    len(path_test_dataset)
)

PATHOLOGY LOADERS CREATED
Train: 249
Test: 54


##Define MLP model

In [59]:
# ============================================================
# PATHOLOGY MLP MODEL
# ============================================================


class PathologyMLP(nn.Module):

    def __init__(
        self,
        input_dim=512,
        dropout=0.3
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),


            nn.Linear(
                256,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),


            nn.Linear(
                128,
                1
            )

        )


    def forward(
        self,
        x
    ):

        return self.network(x)

##Create model

In [60]:
# ============================================================
# CREATE PATHOLOGY MLP
# ============================================================


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


pathology_model = PathologyMLP(
    input_dim=512
)


pathology_model.to(device)


print(
    "Pathology MLP created"
)

Pathology MLP created


##Train MLP

In [63]:
# ============================================================
# TRAIN PATHOLOGY MLP STABLE COX TRAINING
# ============================================================


pathology_model = PathologyMLP(
    input_dim=512
)


pathology_model.to(device)



optimizer = torch.optim.Adam(
    pathology_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)



def cox_loss(
    risks,
    events
):

    order = torch.argsort(
        risks,
        descending=True
    )

    risks = risks[order]

    events = events[order]


    log_cumsum = torch.logcumsumexp(
        risks,
        dim=0
    )


    partial = (
        risks - log_cumsum
    ) * events


    return -partial.sum() / (
        events.sum() + 1e-8
    )



epochs = 100



for epoch in range(epochs):


    pathology_model.train()


    all_risk = []
    all_event = []


    # --------------------------------
    # Forward all training patients
    # --------------------------------

    for batch in path_train_loader:


        x = batch["pathology"].to(device)

        event = batch["event"].to(device)


        risk = pathology_model(
            x
        ).squeeze()


        all_risk.append(
            risk
        )


        all_event.append(
            event
        )



    all_risk = torch.cat(
        all_risk
    )


    all_event = torch.cat(
        all_event
    )


    loss = cox_loss(
        all_risk,
        all_event
    )



    optimizer.zero_grad()


    loss.backward()



    torch.nn.utils.clip_grad_norm_(
        pathology_model.parameters(),
        max_norm=1.0
    )


    optimizer.step()



    if (epoch+1)%10==0:

        print(
            f"Epoch {epoch+1}/{epochs}",
            "Loss:",
            loss.item()
        )

Epoch 10/100 Loss: 4.5373215675354
Epoch 20/100 Loss: 4.473674774169922
Epoch 30/100 Loss: 4.553136825561523
Epoch 40/100 Loss: 4.695192813873291
Epoch 50/100 Loss: 4.561227798461914
Epoch 60/100 Loss: 4.548023223876953
Epoch 70/100 Loss: 4.444969654083252
Epoch 80/100 Loss: 4.350586891174316
Epoch 90/100 Loss: 4.390303134918213
Epoch 100/100 Loss: 4.5751752853393555


In [55]:
print(len(pathology_columns))

print(pathology_columns[:10])

print(pathology_columns[-10:])

515
['pathology_T_stage', 'pathology_N_stage', 'pathology_M_stage', 'pathology_0', 'pathology_1', 'pathology_2', 'pathology_3', 'pathology_4', 'pathology_5', 'pathology_6']
['pathology_502', 'pathology_503', 'pathology_504', 'pathology_505', 'pathology_506', 'pathology_507', 'pathology_508', 'pathology_509', 'pathology_510', 'pathology_511']


##Evaluate pathology MLP

In [64]:
# ============================================================
# PATHOLOGY MLP C-INDEX
# ============================================================


pathology_model.eval()


mlp_risk=[]
mlp_time=[]
mlp_event=[]


with torch.no_grad():

    for batch in path_test_loader:


        x = batch["pathology"].to(device)


        risk = pathology_model(
            x
        ).squeeze()


        mlp_risk.extend(
            risk.cpu().numpy()
        )


        mlp_time.extend(
            batch["survival"].numpy()
        )


        mlp_event.extend(
            batch["event"].numpy()
        )



mlp_cindex = concordance_index_censored(

    np.array(mlp_event).astype(bool),

    np.array(mlp_time),

    np.array(mlp_risk)

)[0]


print("="*70)
print("PATHOLOGY MLP PERFORMANCE")
print("="*70)

print(
    "C-index:",
    mlp_cindex
)

PATHOLOGY MLP PERFORMANCE
C-index: 0.27848101265822783


##Add to comparison table

In [65]:
# ============================================================
# ADD PATHOLOGY MLP RESULT
# ============================================================


comparison = pd.concat(
    [
        comparison,

        pd.DataFrame(
            [
                {
                    "Model":
                    "Pathology MLP",

                    "Modalities":
                    "Pathology Only",

                    "C-index":
                    mlp_cindex
                }
            ]
        )

    ],

    ignore_index=True
)



comparison = comparison.sort_values(
    "C-index",
    ascending=False
).reset_index(drop=True)


display(comparison)

,Model,Modalities,C-index
0,Multimodal Transformer,Clinical + Genomic + Pathology,0.822785
1,CoxPH,Clinical,0.753165
2,RSF,Clinical,0.743671
3,CoxPH,Clinical + Genomic,0.705696
4,CoxPH,Clinical + Genomic + Pathology,0.664557
5,RSF,Clinical + Genomic,0.655063
6,CoxPH,PCA Pathology Only,0.582278
7,RSF,PCA Pathology Only,0.531646
8,RSF,Clinical + Genomic + PCA Pathology,0.325949
9,Pathology MLP,Pathology Only,0.278481


##Fit PCA on pathology features

In [67]:
# ============================================================
# PCA PATHOLOGY FEATURE REDUCTION
# ============================================================


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# pathology columns are already defined

X_train_pathology = train_path[
    pathology_columns
].values


X_test_pathology = test_path[
    pathology_columns
].values



print("="*70)
print("ORIGINAL PATHOLOGY FEATURES")
print("="*70)

print(
    "Train:",
    X_train_pathology.shape
)

print(
    "Test:",
    X_test_pathology.shape
)

ORIGINAL PATHOLOGY FEATURES
Train: (249, 512)
Test: (54, 512)


##Normalize + PCA

In [68]:
# ============================================================
# PCA TRANSFORMATION
# ============================================================


scaler_path = StandardScaler()


X_train_scaled = scaler_path.fit_transform(
    X_train_pathology
)


X_test_scaled = scaler_path.transform(
    X_test_pathology
)



pca_components = 50


pca = PCA(
    n_components=pca_components,
    random_state=42
)



X_train_pca = pca.fit_transform(
    X_train_scaled
)


X_test_pca = pca.transform(
    X_test_scaled
)



print("="*70)
print("PCA COMPLETED")
print("="*70)


print(
    "Train PCA:",
    X_train_pca.shape
)


print(
    "Test PCA:",
    X_test_pca.shape
)



print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)

PCA COMPLETED
Train PCA: (249, 50)
Test PCA: (54, 50)
Explained variance: 0.9630926


##Create PCA datasets

In [69]:
# ============================================================
# PCA PATHOLOGY DATASET
# ============================================================


class PCAPathologyDataset(Dataset):


    def __init__(
        self,
        X,
        dataframe
    ):

        self.X = X

        self.data = dataframe.reset_index(
            drop=True
        )


    def __len__(self):

        return len(self.data)



    def __getitem__(
        self,
        idx
    ):


        return {

            "pathology":
            torch.tensor(
                self.X[idx],
                dtype=torch.float32
            ),


            "event":
            torch.tensor(
                self.data.iloc[idx]["event"],
                dtype=torch.float32
            ),


            "survival":
            torch.tensor(
                self.data.iloc[idx]["survival_time"],
                dtype=torch.float32
            )

        }

##Create loaders

In [70]:
# ============================================================
# PCA PATHOLOGY LOADERS
# ============================================================


pca_train_dataset = PCAPathologyDataset(
    X_train_pca,
    train_path
)


pca_test_dataset = PCAPathologyDataset(
    X_test_pca,
    test_path
)



pca_train_loader = DataLoader(
    pca_train_dataset,
    batch_size=16,
    shuffle=True
)



pca_test_loader = DataLoader(
    pca_test_dataset,
    batch_size=16,
    shuffle=False
)



print("="*70)
print("PCA LOADERS CREATED")
print("="*70)


print(
    len(pca_train_dataset)
)

print(
    len(pca_test_dataset)
)

PCA LOADERS CREATED
249
54


##PCA MLP model

In [71]:
# ============================================================
# PCA PATHOLOGY MLP
# ============================================================


class PCAPathologyMLP(nn.Module):


    def __init__(
        self,
        input_dim=50,
        dropout=0.3
    ):

        super().__init__()


        self.model = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.GELU(),

            nn.Dropout(dropout),


            nn.Linear(
                64,
                32
            ),

            nn.GELU(),


            nn.Dropout(dropout),


            nn.Linear(
                32,
                1
            )

        )


    def forward(
        self,
        x
    ):

        return self.model(x)

##Train PCA MLP

In [72]:
# ============================================================
# TRAIN PCA PATHOLOGY MLP
# ============================================================


pca_model = PCAPathologyMLP(
    input_dim=50
)


pca_model.to(device)



optimizer = torch.optim.Adam(
    pca_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)



epochs = 100



for epoch in range(epochs):


    pca_model.train()


    risks_all=[]
    events_all=[]



    for batch in pca_train_loader:


        x = batch["pathology"].to(device)

        event = batch["event"].to(device)



        risk = pca_model(
            x
        ).squeeze()



        risks_all.append(
            risk
        )


        events_all.append(
            event
        )



    risks_all = torch.cat(
        risks_all
    )


    events_all = torch.cat(
        events_all
    )



    loss = cox_loss(
        risks_all,
        events_all
    )



    optimizer.zero_grad()

    loss.backward()


    torch.nn.utils.clip_grad_norm_(
        pca_model.parameters(),
        1.0
    )


    optimizer.step()



    if (epoch+1)%10==0:

        print(
            f"Epoch {epoch+1}/{epochs}",
            "Loss:",
            loss.item()
        )

Epoch 10/100 Loss: 4.857790470123291
Epoch 20/100 Loss: 4.7714056968688965
Epoch 30/100 Loss: 4.841338634490967
Epoch 40/100 Loss: 4.924212455749512
Epoch 50/100 Loss: 5.043684005737305
Epoch 60/100 Loss: 4.659931659698486
Epoch 70/100 Loss: 4.84634256362915
Epoch 80/100 Loss: 4.816947937011719
Epoch 90/100 Loss: 4.962374687194824
Epoch 100/100 Loss: 4.935888767242432


##Evaluate PCA MLP

In [73]:
# ============================================================
# PCA MLP C-INDEX
# ============================================================


pca_model.eval()


pca_risk=[]
pca_time=[]
pca_event=[]



with torch.no_grad():


    for batch in pca_test_loader:


        x = batch["pathology"].to(device)


        risk = pca_model(
            x
        ).squeeze()



        pca_risk.extend(
            risk.cpu().numpy()
        )


        pca_time.extend(
            batch["survival"].numpy()
        )


        pca_event.extend(
            batch["event"].numpy()
        )



pca_mlp_cindex = concordance_index_censored(

    np.array(pca_event).astype(bool),

    np.array(pca_time),

    np.array(pca_risk)

)[0]



print("="*70)
print("PCA PATHOLOGY MLP PERFORMANCE")
print("="*70)


print(
    "C-index:",
    pca_mlp_cindex
)

PCA PATHOLOGY MLP PERFORMANCE
C-index: 0.3322784810126582


##Add to comparison table

In [74]:
# ============================================================
# ADD PCA MLP RESULT
# ============================================================


comparison = pd.concat(

    [

        comparison,


        pd.DataFrame(
            [
                {
                    "Model":
                    "PCA Pathology MLP",


                    "Modalities":
                    "PCA Pathology Only",


                    "C-index":
                    pca_mlp_cindex
                }
            ]
        )

    ],

    ignore_index=True
)



comparison = comparison.sort_values(
    "C-index",
    ascending=False
).reset_index(drop=True)



display(comparison)

,Model,Modalities,C-index
0,Multimodal Transformer,Clinical + Genomic + Pathology,0.822785
1,CoxPH,Clinical,0.753165
2,RSF,Clinical,0.743671
3,CoxPH,Clinical + Genomic,0.705696
4,CoxPH,Clinical + Genomic + Pathology,0.664557
5,RSF,Clinical + Genomic,0.655063
6,CoxPH,PCA Pathology Only,0.582278
7,RSF,PCA Pathology Only,0.531646
8,PCA Pathology MLP,PCA Pathology Only,0.332278
9,RSF,Clinical + Genomic + PCA Pathology,0.325949


##